# YOLOE-v8-S GPU benchmark
Select Runtime > Change runtime type > T4 GPU, then run all cells in order.
Upload the accompanying uav_yoloe_colab.zip. The original sample 1.png is included.
Uses YOLOE-v8-S at 1280 with the project catalogue. Reports loading, first inference, warmed inference and read/detect/save timing separately. GPU results are available only after running this notebook.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU under Runtime > Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
bundle = next((name for name in uploaded if name.endswith('.zip')), None)
assert bundle, 'Upload the project ZIP first.'
project = Path('/content/uav_yoloe')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    for member in archive.infolist():
        target = (project / member.filename).resolve()
        assert target.is_relative_to(project.resolve()), 'Invalid ZIP path'
    archive.extractall(project)
os.chdir(project)
print('Project:', project)


In [ ]:
%pip -q install -r lan_chiem/requirements-yoloe.txt


In [ ]:
# Upload an image optionally; FP32 is the default for comparison.
UPLOAD_IMAGE = False
IMAGE_SIZE = 1280
FP16 = False  # Set True to test GPU half precision after FP32.
RUNS = 10
WARMUP = 3
IMAGE = project / 'lan_chiem/sample_data/1.png'
if UPLOAD_IMAGE:
    images = files.upload()
    image_name = next(iter(images))
    IMAGE = Path(image_name).resolve()
assert IMAGE.is_file(), f'Missing image: {IMAGE}'


In [ ]:
import time, sys, json, hashlib
import numpy as np
from PIL import Image
sys.path.insert(0, str(project))
from lan_chiem.yoloe_detector import YOLOEDetector
from lan_chiem.visualization import draw_detections

def sync():
    torch.cuda.synchronize()

def timed(fn):
    sync()
    start = time.perf_counter()
    value = fn()
    sync()
    return value, time.perf_counter() - start

detector = YOLOEDetector(device='cuda', imgsz=IMAGE_SIZE)
detector.precision = 'fp16' if FP16 else 'fp32'
_, load_seconds = timed(detector.load_model)
# Upload an image optionally; FP32 is the default for comparison.
image = Image.open(IMAGE).convert('RGB')
first, first_seconds = timed(lambda: detector.predict(image))
print(f'Model and embedding setup (may include downloads): {load_seconds:.3f}s')
print(f'First inference: {first_seconds:.3f}s')
for _ in range(WARMUP):
    detector.predict(image)
sync()
latencies = []
for _ in range(RUNS):
    result, elapsed = timed(lambda: detector.predict(image))
    latencies.append(elapsed)
print(f'Warm inference: median={np.median(latencies):.4f}s, mean={np.mean(latencies):.4f}s')
print('Detection count:', result['summary']['total_predicted_detections'])


In [ ]:
output = project / 'outputs/yoloe_gpu_1280'
output.mkdir(parents=True, exist_ok=True)

def analyze_and_save():
    result = detector.predict(IMAGE)
    pil = result.pop('_image_pil')
    annotated = draw_detections(pil, result['object_detections'], label_mode='short')
    annotated.save(output / 'annotated.jpg', quality=95)
    (output / 'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    return result

end_to_end = []
for _ in range(RUNS):
    final_result, elapsed = timed(analyze_and_save)
    end_to_end.append(elapsed)
summary = {
    'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
    'model': detector.weights, 'imgsz': IMAGE_SIZE, 'precision': detector.precision,
    'image_sha256': hashlib.sha256(IMAGE.read_bytes()).hexdigest(),
    'image_size': list(image.size), 'runs': RUNS, 'warmup': WARMUP,
    'load_seconds': load_seconds, 'first_inference_seconds': first_seconds,
    'inference_median_seconds': float(np.median(latencies)),
    'inference_mean_seconds': float(np.mean(latencies)),
    'end_to_end_median_seconds': float(np.median(end_to_end)),
    'end_to_end_mean_seconds': float(np.mean(end_to_end)),
    'inference_samples_seconds': latencies, 'end_to_end_samples_seconds': end_to_end,
    'detections': final_result['summary'],
}
(output / 'benchmark.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({k:v for k,v in summary.items() if 'samples' not in k}, ensure_ascii=False, indent=2))
print(f"GPU read/detect/save median: {summary['end_to_end_median_seconds']:.3f}s/image")
print(f"Speedup versus approximate CPU 2s baseline: {2 / summary['end_to_end_median_seconds']:.1f}x")
print('Warmed end-to-end measurement excludes Python startup and imports.')
from IPython.display import display
display(Image.open(output / 'annotated.jpg'))


In [ ]:
import shutil
archive_path = shutil.make_archive('/content/yoloe_gpu_results', 'zip', output)
files.download(archive_path)
